# 🌲 تدريب نموذج YOLOv8 Nano لرصد وعد أشجار المزارع والمصدات (Tree Crown Detection)
### تدريب فائق السرعة والخفة في أقل من 10 دقائق على Google Colab (GPU T4)

هذا الدفتر مجهز بالكامل لتدريب نموذج الذكاء الاصطناعي YOLOv8 Nano وتصدير ملف الأوزان النهائي **`best.pt`** لدمجه مباشرة في تطبيق تدقيق المزارع.

## 1️⃣ تثبيت مكتبة Ultralytics وفحص كارت الشاشة GPU

In [ ]:
!pip install -q ultralytics
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2️⃣ تحميل البيانات من Roboflow أو استخدام البيانات الجاهزة
إذا كان لديك كود Roboflow، ضعه هنا مباشرة:

In [ ]:
# خيار أ: تحميل مباشر من Roboflow (استبدل بمفتاحك واسم مشروعك إن وجد)
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("workspace-name").project("farm-trees")
# dataset = project.version(1).download("yolov8")

# خيار ب: إنشاء ملف data.yaml محلياً
import yaml

data_yaml = """
train: dataset/images/train
val: dataset/images/val
names:
  0: tree
"""
with open('data.yaml', 'w') as f:
    f.write(data_yaml)
print("data.yaml created successfully!")

## 3️⃣ بدء التدريب فائق السرعة باستخدام YOLOv8 Nano
يستغرق التدريب حوالي 5-8 دقائق فقط على كارت T4 المجاني.

In [ ]:
from ultralytics import YOLO

# تحميل النموذج المبدئي خفيف الوزن (6 ميجابايت فقط)
model = YOLO('yolov8n.pt')

# بدء تدريب الأشجار لـ 30 حقبة (Epochs)
results = model.train(
    data='data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)
print("✅ اكتمل التدريب بنجاح!")

## 4️⃣ تحميل ملف الأوزان `best.pt` النهائي لجهازك
الملف الناتج هو **`best.pt`** — بمجرد وضعه في مجلد التطبيق الرئيسي، سيقوم التطبيق بتفعيله وقراءته تلقائياً!

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

save_dir = Path(model.trainer.save_dir)
best_weights = save_dir / 'weights' / 'best.pt'
shutil.copy(best_weights, 'best.pt')

print("📥 جاري تنزيل ملف best.pt للكمبيوتر...")
files.download('best.pt')